In [47]:
import os

os.environ["APP_ID"] = "bf2ea6b3"
os.environ["API_KEY"] = "1ace9ffbad536d91f99ae742101b1c5c"

app_id = os.getenv("APP_ID")
api_key = os.getenv("API_KEY")

print("APP_ID:", app_id)
print("API_KEY:", api_key)


APP_ID: bf2ea6b3
API_KEY: 1ace9ffbad536d91f99ae742101b1c5c


In [48]:
import requests
import time
from dotenv import load_dotenv
import os

# Load environment variables from .env file
load_dotenv()

# Get API credentials from environment variables
app_id = os.getenv("APP_ID")
api_key = os.getenv("API_KEY")

# List of station codes
station_ids = ["crs:RMD", "WAT", "LYM"]  # Replace with actual station codes

# Function to fetch timetable data for a list of stations
def fetch_station_timetables(app_id, api_key, station_ids):
    if not app_id or not api_key:
        print("API credentials are missing. Check your .env file.")
        return {}

    timetable_data = {}

    for station_id in station_ids:
        # Construct the API URL for the current station
        url = f"https://transportapi.com/v3/uk/train/station_timetables/{station_id}.json"
        
        # Define request parameters
        params = {
            "app_key": api_key,
            "app_id": app_id
        }
        
        # Make the API request
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            timetable_data[station_id] = data
            print(f"Successfully fetched data for station: {station_id}")
        else:
            print(f"Failed to fetch data for station: {station_id}. Status Code: {response.status_code}")
        
        # Respect API rate limits
        time.sleep(1)

    return timetable_data

# Fetch data for the stations
timetable_data = fetch_station_timetables(app_id, api_key, station_ids)

# Print the fetched data
print("Collected Timetable Data:")
print(timetable_data)

Successfully fetched data for station: crs:RMD
Successfully fetched data for station: WAT
Successfully fetched data for station: LYM
Collected Timetable Data:
{'crs:RMD': {'date': '2024-12-03', 'time_of_day': '15:47', 'request_time': '2024-12-03T15:47:54+00:00', 'station_name': 'Richmond', 'station_code': 'crs:RMD', 'departures': {'all': [{'mode': 'train', 'service': '24671405', 'train_uid': 'L60498', 'platform': '2', 'operator': 'SW', 'operator_name': 'South Western Railway', 'aimed_departure_time': '15:49', 'aimed_arrival_time': '15:48', 'aimed_pass_time': None, 'origin_name': 'London Waterloo', 'destination_name': 'London Waterloo', 'source': 'ATOC', 'category': 'OO', 'service_timetable': {'id': 'https://transportapi.com/v3/uk/train/service_timetables/L60498:2024-12-03.json?app_id=bf2ea6b3&app_key=1ace9ffbad536d91f99ae742101b1c5c'}}, {'mode': 'train', 'service': '24682004', 'train_uid': 'C13620', 'platform': '6', 'operator': 'LT', 'operator_name': 'London Underground', 'aimed_depart

In [72]:
import requests
import time
from dotenv import load_dotenv
import os
from kafka import KafkaProducer
import json

# Load environment variables from .env file
load_dotenv()

# Get API credentials from environment variables
app_id = os.getenv("APP_ID")
api_key = os.getenv("API_KEY")

# Kafka setup
kafka_server = 'localhost:9092'  # Kafka broker
topic_name = 'train_timetables'  # Kafka topic to send data to

# List of station codes
station_ids = ["crs:RMD", "WAT", "LYM"]  # Replace with actual station codes

# Function to fetch timetable data for a list of stations
def fetch_station_timetables(app_id, api_key, station_ids):
    if not app_id or not api_key:
        print("API credentials are missing. Check your .env file.")
        return {}

    timetable_data = {}

    # Create a Kafka producer
    producer = KafkaProducer(bootstrap_servers=kafka_server, 
                             value_serializer=lambda v: json.dumps(v).encode('utf-8'))

    for station_id in station_ids:
        # Construct the API URL for the current station
        url = f"https://transportapi.com/v3/uk/train/station_timetables/{station_id}.json"
        
        # Define request parameters
        params = {
            "app_key": api_key,
            "app_id": app_id
        }
        
        # Make the API request
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            timetable_data[station_id] = data
            print(f"Successfully fetched data for station: {station_id}")

            # Send the fetched data to Kafka
            producer.send(topic_name, value={station_id: data})
            print(f"Sent data to Kafka for station: {station_id}")
        else:
            print(f"Failed to fetch data for station: {station_id}. Status Code: {response.status_code}")
        
        # Respect API rate limits
        time.sleep(1)

    # Close the Kafka producer
    producer.flush()
    producer.close()

    return timetable_data

# Fetch data for the stations and stream to Kafka
timetable_data = fetch_station_timetables(app_id, api_key, station_ids)

# Print the fetched data
print("Collected Timetable Data:")
print(timetable_data)


Successfully fetched data for station: crs:RMD
Sent data to Kafka for station: crs:RMD
Successfully fetched data for station: WAT
Sent data to Kafka for station: WAT
Successfully fetched data for station: LYM
Sent data to Kafka for station: LYM
Collected Timetable Data:
{'crs:RMD': {'date': '2024-12-04', 'time_of_day': '16:13', 'request_time': '2024-12-04T16:13:34+00:00', 'station_name': 'Richmond', 'station_code': 'crs:RMD', 'departures': {'all': [{'mode': 'train', 'service': '24672104', 'train_uid': 'L58621', 'platform': '2', 'operator': 'SW', 'operator_name': 'South Western Railway', 'aimed_departure_time': '16:15', 'aimed_arrival_time': '16:15', 'aimed_pass_time': None, 'origin_name': 'Reading', 'destination_name': 'London Waterloo', 'source': 'ATOC', 'category': 'OO', 'service_timetable': {'id': 'https://transportapi.com/v3/uk/train/service_timetables/L58621:2024-12-04.json?app_id=bf2ea6b3&app_key=1ace9ffbad536d91f99ae742101b1c5c'}}, {'mode': 'train', 'service': '24682004', 'train

In [80]:
import requests
import time
from dotenv import load_dotenv
import os
from kafka import KafkaProducer
import json
import pandas as pd

# Load environment variables from .env file
load_dotenv()

# Get API credentials from environment variables
app_id = os.getenv("APP_ID")
api_key = os.getenv("API_KEY")

# Kafka setup
kafka_server = 'localhost:9092'  # Kafka broker
topic_name = 'train_timetables'  # Kafka topic to send data to

# List of station codes
station_ids = ["crs:RMD", "WAT", "LYM"]  # Replace with actual station codes

# Function to fetch timetable data for a list of stations
def fetch_station_timetables(app_id, api_key, station_ids):
    if not app_id or not api_key:
        print("API credentials are missing. Check your .env file.")
        return {}

    timetable_data = {}

    # Create a Kafka producer
    producer = KafkaProducer(bootstrap_servers=kafka_server, 
                             value_serializer=lambda v: json.dumps(v).encode('utf-8'))

    for station_id in station_ids:
        # Construct the API URL for the current station
        url = f"https://transportapi.com/v3/uk/train/station_timetables/{station_id}.json"
        
        # Define request parameters
        params = {
            "app_key": api_key,
            "app_id": app_id
        }
        
        # Make the API request
        response = requests.get(url, params=params)
        
        if response.status_code == 200:
            data = response.json()
            timetable_data[station_id] = data
            print(f"Successfully fetched data for station: {station_id}")

            # Send the fetched data to Kafka
            producer.send(topic_name, value={station_id: data})
            print(f"Sent data to Kafka for station: {station_id}")
        else:
            print(f"Failed to fetch data for station: {station_id}. Status Code: {response.status_code}")
        
        # Respect API rate limits
        time.sleep(1)

    # Close the Kafka producer
    producer.flush()
    producer.close()

    return timetable_data

# Fetch data for the stations and stream to Kafka
timetable_data = fetch_station_timetables(app_id, api_key, station_ids)

# Print the fetched data
print("Collected Timetable Data:")
print(timetable_data)

# Convert fetched timetable data into a pandas DataFrame
# Flatten the data (in case it's nested) and create a DataFrame
df_timetable = pd.json_normalize(timetable_data)

# View the DataFrame
print(df_timetable)

Successfully fetched data for station: crs:RMD
Sent data to Kafka for station: crs:RMD
Successfully fetched data for station: WAT
Sent data to Kafka for station: WAT
Successfully fetched data for station: LYM
Sent data to Kafka for station: LYM
Collected Timetable Data:
{'crs:RMD': {'date': '2024-12-04', 'time_of_day': '17:09', 'request_time': '2024-12-04T17:09:19+00:00', 'station_name': 'Richmond', 'station_code': 'crs:RMD', 'departures': {'all': [{'mode': 'train', 'service': '24671305', 'train_uid': 'L61660', 'platform': '1', 'operator': 'SW', 'operator_name': 'South Western Railway', 'aimed_departure_time': '17:12', 'aimed_arrival_time': '17:11', 'aimed_pass_time': None, 'origin_name': 'London Waterloo', 'destination_name': 'London Waterloo', 'source': 'ATOC', 'category': 'OO', 'service_timetable': {'id': 'https://transportapi.com/v3/uk/train/service_timetables/L61660:2024-12-04.json?app_id=bf2ea6b3&app_key=1ace9ffbad536d91f99ae742101b1c5c'}}, {'mode': 'train', 'service': '24672104'

#TRANSFORMATION

In [82]:
# Flatten and clean the timetable data
cleaned_data = []

for station_id, station_data in timetable_data.items():
    if 'departures' in station_data and 'all' in station_data['departures']:
        departures = station_data['departures']['all']
        for departure in departures:
            cleaned_entry = {
                'station_id': station_id,
                'train_uid': departure.get('train_uid'),
                'scheduled_departure_time': departure.get('aimed_departure_time'),
                'destination_name': departure.get('destination', [{}])[0].get('name'),
                'platform': departure.get('platform'),
                'operator_name': departure.get('operator_name'),
            }
            cleaned_data.append(cleaned_entry)

# Create a pandas DataFrame
df_timetable = pd.DataFrame(cleaned_data)

# Handle missing values
df_timetable.fillna("Unknown", inplace=True)

# Rename columns for clarity
df_timetable.rename(columns={
    'station_id': 'Station ID',
    'train_uid': 'Train UID',
    'scheduled_departure_time': 'Departure Time',
    'destination_name': 'Destination',
    'platform': 'Platform',
    'operator_name': 'Operator'
}, inplace=True)

# View the cleaned data
print("Cleaned Timetable Data:")
print(df_timetable)

# Save the DataFrame to a CSV file for further analysis
df_timetable.to_csv("cleaned_timetable_data.csv", index=False)

Cleaned Timetable Data:
    Station ID Train UID Departure Time Destination Platform  \
0      crs:RMD    L61660          17:12     Unknown        1   
1      crs:RMD    L58639          17:15     Unknown        2   
2      crs:RMD    C13725          17:17     Unknown        6   
3      crs:RMD    L60505          17:19     Unknown        2   
4      crs:RMD    P90783          17:20     Unknown        5   
..         ...       ...            ...         ...      ...   
103        LYM    G43924          18:00     Unknown  Unknown   
104        LYM    G45150          18:16     Unknown  Unknown   
105        LYM    G45185          18:30     Unknown  Unknown   
106        LYM    G45126          18:46     Unknown  Unknown   
107        LYM    G44111          19:00     Unknown  Unknown   

                  Operator  
0    South Western Railway  
1    South Western Railway  
2       London Underground  
3    South Western Railway  
4        London Overground  
..                     ...  
103 

##DATA VALIDATION

In [116]:
import great_expectations as ge
import pandas as pd

In [ ]:
import great_expectations as ge
from great_expectations.data_context import get_context

# Path to your Great Expectations directory
GE_DIR = "/Users/machine/Velocity_railway/gx"

# Initialize the Great Expectations context using get_context() method
context = get_context(context_root_dir=GE_DIR)

# Proceed with your DAG task function
def load_and_validate_data():
    # Load your DataFrame
    df_timetable = pd.read_csv("cleaned_timetable_data.csv")

    # Create a Great Expectations Dataset (Batch)
    batch = ge.dataset.PandasDataset(df_timetable)

    # Add expectations to validate the data
    batch.expect_column_values_to_be_in_set("column_name", ["value1", "value2"])

    # Run validation and print the results
    validation_results = batch.validate()
    
    # Save or process the validation results (you can also log or store them)
    print("Validation Results:", validation_results)
    return validation_results


In [ ]:
import great_expectations as ge
from great_expectations.data_context import get_context

# Path to your Great Expectations project directory
GE_DIR = "/Users/machine/Velocity_railway/gx"

# Initialize the Great Expectations context
context = get_context(context_root_dir=GE_DIR)

# Print out the context configuration for debugging
print(context.config)